# Customer Support Resolution Center

**Mode:** Offline  
**Level:** Capstone  
**Estimated time:** 60 minutes

Support resolution is a coordination system: current service state, customer
history, technical evidence, policy, tone, and approval all need different
owners. This notebook follows one enterprise incident from intake to closure.


## Problem and outcome

Acme Freight's scheduled export is two hours late after a release regression.
The customer has experienced this before, asks for a 25 percent service credit,
and explicitly dislikes generic cache-clearing advice. The evidence includes a
stale article that recommends exactly that bad response.

The team must diagnose the live incident, respect the customer's history,
calculate a policy-bounded credit, obtain explicit reviewer input, and create an
auditable closed-case record.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "examples" / "notebooks"]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, setup_case_study, show_artifact, show_callout,
    show_flow, show_json, show_message_graph, show_roles, show_scorecard,
    show_spore, show_timeline, stage,
)

setup_case_study("Customer Support Resolution Center")


## Course prerequisites

Complete the Reef pipeline, parallel fan-in, tools, and memory lessons. This
case study uses an explicit reviewer Spore as a deterministic human boundary.
It does **not** claim to be ModelRuntime HITL; true provider-driven interruption
and resume is taught in course 10 and used by the Marketing Studio.


## Architecture and responsibilities

```text
case_opened → triage_decision
            → context_snapshot + knowledge_evidence + policy_decision
            → specialist_diagnosis → resolution_draft → quality_decision
            → escalation_request → reviewer_decision → case_closed
```

All messages carry the case ID and one correlation ID. The state machine records
`OPEN`, `TRIAGED`, `DIAGNOSED`, `REVIEW_REQUIRED`, `REVIEWED`, and `CLOSED`.


In [ ]:
show_flow(
    ("Intake", "urgency and sentiment", "human"),
    ("Context fan-out", "account, knowledge, policy", "agent"),
    ("Diagnosis", "current service evidence", "tool"),
    ("Quality", "reject stale guidance", "agent"),
    ("Reviewer", "bounded credit decision", "human"),
    ("Closure", "auditable response", "spore"),
)
show_roles([
    ("Intake and sentiment triage", "Classify urgency and customer impact", "agent"),
    ("Customer-context retriever", "Retrieve history and durable preferences", "agent"),
    ("Knowledge specialist", "Separate current from stale guidance", "agent"),
    ("Technical specialist", "Join evidence and diagnose the regression", "agent"),
    ("Policy specialist", "Calculate the permitted credit boundary", "agent"),
    ("Resolution coordinator", "Draft, escalate, revise, and close", "agent"),
    ("Customer advocate", "Reject unsafe or context-free responses", "agent"),
    ("Escalation reviewer", "Provide explicit deterministic approval", "human"),
])


## Design decisions

- Account records, service status, knowledge articles, entitlement, and credit
  calculations are registered Praval tools.
- Short-term case facts stay in the correlated trail; durable communication
  preferences live in customer memory.
- Quality deliberately rejects revision 0 because it cites stale guidance and
  ignores prior context.
- The reviewer can approve only within the entitlement tool's cap.


## Implementation

The contract below gives every handoff enough identity to reconstruct the case.
The fixture remains external, so tools rather than handler constants own facts.


In [ ]:
import json

from praval import Agent, agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef
from praval.core.tool_registry import reset_tool_registry
from praval.tools import tool

reset_reef()
reset_tool_registry()

case_path = find_example_asset("notebooks/fixtures/support_case.json")
support_data = json.loads(case_path.read_text(encoding="utf-8"))
REQUIRED = {"type", "domain_id", "correlation_id", "producer", "stage", "status", "payload"}


def case_message(message_type, producer, stage_name, status, payload):
    message = {
        "type": message_type, "domain_id": support_data["case"]["id"],
        "correlation_id": "support-case-4815", "producer": producer,
        "stage": stage_name, "status": status, "payload": payload,
    }
    assert set(message) == REQUIRED
    return message


message_trail, received_spores, tool_events = [], [], []
case_states, case_records = ["OPEN"], []


In [ ]:
@tool("support_account_lookup", description="Return the bounded account record.",
      category="support", shared=True)
def account_lookup(account_id: str) -> dict:
    if account_id != support_data["case"]["account_id"]:
        raise ValueError("unknown account")
    tool_events.append({"tool": "account_lookup", "account_id": account_id})
    return dict(support_data["account"])


@tool("support_knowledge_search", description="Search current and stale support guidance.",
      category="support", shared=True)
def knowledge_search(query: str) -> list:
    tool_events.append({"tool": "knowledge_search", "query": query})
    return [dict(item) for item in support_data["knowledge_articles"]]


@tool("support_service_status", description="Return current component status.",
      category="support", shared=True)
def service_status(component: str) -> dict:
    status = support_data["service_status"]
    if component != status["component"]:
        raise ValueError("unknown component")
    tool_events.append({"tool": "service_status", "component": component})
    return dict(status)


@tool("support_entitlement", description="Return account and approval credit limits.",
      category="support", shared=True)
def entitlement_check(account_id: str) -> dict:
    account = account_lookup.execute_as_tool(account_id)
    result = {"plan": account["plan"], "cap_percent": account["credit_cap_percent"],
              **support_data["policy"]}
    tool_events.append({"tool": "entitlement_check", "account_id": account_id})
    return result


@tool("support_credit", description="Calculate a policy-bounded service credit.",
      category="support", shared=True)
def credit_calculation(monthly_fee: float, requested_percent: float, cap_percent: float) -> dict:
    approved = min(requested_percent, cap_percent)
    result = {"requested_percent": requested_percent, "approved_percent": approved,
              "amount_gbp": round(monthly_fee * approved / 100, 2)}
    tool_events.append({"tool": "credit_calculation", **result})
    return result


In [ ]:
customer_memory = Agent(
    "customer-context-memory", provider="notebook-lifecycle",
    model="notebook-lifecycle-model", memory_enabled=True,
    memory_config={"backend": "memory"},
)
for preference in support_data["account"]["preferences"]:
    customer_memory.remember(f"Durable preference: {preference}", 0.9)
for history in support_data["account"]["history"]:
    customer_memory.remember(f"Prior case context: {history}", 0.8)


def remember_spore(spore):
    received_spores.append(spore)
    message_trail.append(dict(spore.knowledge))


@agent("support-intake", responds_to=["case_opened"], auto_broadcast=False)
def intake_agent(spore):
    remember_spore(spore)
    case_states.append("TRIAGED")
    message = spore.knowledge["payload"]["message"].lower()
    triage = {"priority": "P1", "sentiment": "frustrated",
              "signals": [word for word in ("again", "broke", "credit") if word in message]}
    broadcast(case_message("triage_decision", "support-intake", "triage", "complete", triage))


@agent("customer-context-retriever", responds_to=["triage_decision"],
       tools=["support_account_lookup"], auto_broadcast=False)
def context_agent(spore):
    remember_spore(spore)
    account = account_lookup.execute_as_tool(support_data["case"]["account_id"])
    memories = [entry.content for entry in customer_memory.recall("preference prior", limit=20)]
    broadcast(case_message("context_snapshot", "customer-context-retriever", "context",
                           "complete", {"account": account, "memories": memories}))


In [ ]:
@agent("knowledge-specialist", responds_to=["triage_decision"],
       tools=["support_knowledge_search"], auto_broadcast=False)
def knowledge_agent(spore):
    remember_spore(spore)
    articles = knowledge_search.execute_as_tool("scheduled export delay")
    current = [item for item in articles if item["status"] == "current"]
    stale = [item for item in articles if item["status"] == "stale"]
    broadcast(case_message("knowledge_evidence", "knowledge-specialist", "knowledge",
                           "complete", {"current": current, "stale": stale}))


@agent("policy-specialist", responds_to=["triage_decision"],
       tools=["support_entitlement", "support_credit"], auto_broadcast=False)
def policy_agent(spore):
    remember_spore(spore)
    account_id = support_data["case"]["account_id"]
    entitlement = entitlement_check.execute_as_tool(account_id)
    monthly_fee = support_data["account"]["annual_value_gbp"] / 12
    credit = credit_calculation.execute_as_tool(
        monthly_fee, support_data["case"]["requested_credit_percent"], entitlement["cap_percent"]
    )
    broadcast(case_message("policy_decision", "policy-specialist", "policy",
                           "review_required", {"entitlement": entitlement, "credit": credit}))


diagnosis_inputs = {}


@agent("technical-specialist", responds_to=["context_snapshot", "knowledge_evidence", "policy_decision"],
       tools=["support_service_status"], auto_broadcast=False)
def technical_agent(spore):
    remember_spore(spore)
    diagnosis_inputs[spore.knowledge["type"]] = spore.knowledge["payload"]
    if len(diagnosis_inputs) != 3:
        return
    status = service_status.execute_as_tool("scheduled-exports-eu-west")
    diagnosis = {"cause": status["cause"], "incident": status["incident"],
                 "mitigation": status["mitigation"], "next_update_at": status["next_update_at"],
                 "context": diagnosis_inputs}
    case_states.append("DIAGNOSED")
    broadcast(case_message("specialist_diagnosis", "technical-specialist", "diagnosis",
                           "complete", diagnosis))


In [ ]:
@agent("resolution-coordinator", responds_to=["specialist_diagnosis", "quality_decision", "reviewer_decision"],
       auto_broadcast=False)
def resolution_agent(spore):
    remember_spore(spore)
    message_type = spore.knowledge["type"]
    if message_type == "specialist_diagnosis":
        draft = {
            "revision": 0, "diagnosis": spore.knowledge["payload"]["cause"],
            "knowledge_id": "KB-122",
            "customer_response": "Clear your browser cache and retry the export.",
            "used_customer_context": False,
        }
        broadcast(case_message("resolution_draft", "resolution-coordinator", "draft",
                               "needs_review", draft))
    elif message_type == "quality_decision":
        case_states.append("REVIEW_REQUIRED")
        escalation = {"reason": spore.knowledge["payload"]["reasons"],
                      "requested_credit_percent": support_data["case"]["requested_credit_percent"],
                      "policy_cap_percent": support_data["account"]["credit_cap_percent"]}
        broadcast(case_message("escalation_request", "resolution-coordinator", "escalation",
                               "pending_reviewer", escalation))
    else:
        reviewer = spore.knowledge["payload"]
        current = support_data["knowledge_articles"][1]
        record = {
            "case_id": support_data["case"]["id"], "state": "CLOSED",
            "diagnosis": support_data["service_status"]["cause"],
            "evidence": [support_data["service_status"]["incident"], current["id"]],
            "policy_rationale": support_data["policy"]["incident_credit_rule"],
            "approved_action": reviewer,
            "customer_response": (
                "Scheduled exports are degraded by incident INC-2026-017; rollback is in progress. "
                "We will send the next update at 10:30 UTC, copy your finance operations lead, "
                "and apply the approved 15 percent service credit. No browser action is required."
            ),
            "escalation_trail": ["quality rejection", "explicit reviewer Spore", "bounded approval"],
        }
        case_records.append(record)
        case_states.extend(["REVIEWED", "CLOSED"])
        customer_memory.remember("CASE-4815 closed with a 15 percent approved credit", 1.0)
        broadcast(case_message("case_closed", "resolution-coordinator", "closure", "complete", record))


@agent("customer-advocate", responds_to=["resolution_draft", "case_closed"], auto_broadcast=False)
def quality_agent(spore):
    remember_spore(spore)
    if spore.knowledge["type"] == "case_closed":
        return
    draft = spore.knowledge["payload"]
    reasons = []
    if draft["knowledge_id"] == "KB-122":
        reasons.append("stale knowledge article")
    if not draft["used_customer_context"]:
        reasons.append("prior customer context omitted")
    assert reasons
    broadcast(case_message("quality_decision", "customer-advocate", "quality",
                           "rejected", {"decision": "revise", "reasons": reasons}))


In [ ]:
reviewer_input = {
    "reviewer": "support-duty-manager", "decision": "approve_with_edit",
    "approved_credit_percent": 15, "requested_credit_percent": 25,
    "condition": "Use current incident guidance and honor communication preferences.",
}


@agent("escalation-reviewer", responds_to=["escalation_request"], auto_broadcast=False)
def escalation_reviewer(spore):
    remember_spore(spore)
    request = spore.knowledge["payload"]
    assert reviewer_input["approved_credit_percent"] <= request["policy_cap_percent"]
    stage("escalation reviewer", "explicit decision", reviewer_input["decision"], spore)
    broadcast(case_message("reviewer_decision", "escalation-reviewer", "human_boundary",
                           "approved", reviewer_input))


## End-to-end run

The initial case fans out after triage. The technical specialist waits for
context, knowledge, and policy terminal messages before diagnosing. The first
draft is rejected, so closure cannot occur until the reviewer Spore arrives.


In [ ]:
agents = (
    intake_agent, context_agent, knowledge_agent, policy_agent,
    technical_agent, resolution_agent, quality_agent, escalation_reviewer,
)
start_agents(
    *agents,
    initial_data=case_message(
        "case_opened", "customer", "intake", "open",
        {"message": support_data["case"]["message"],
         "requested_credit_percent": support_data["case"]["requested_credit_percent"]},
    ),
    channel="support-resolution-center",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=30)

assert case_states == ["OPEN", "TRIAGED", "DIAGNOSED", "REVIEW_REQUIRED", "REVIEWED", "CLOSED"]
assert len(case_records) == 1 and case_records[0]["state"] == "CLOSED"
assert "KB-418" in case_records[0]["evidence"] and "KB-122" not in case_records[0]["evidence"]
assert {item["correlation_id"] for item in message_trail} == {"support-case-4815"}


## Inspect the system

The record below separates diagnosis, evidence, policy rationale, approved
action, customer language, and escalation trail. That separation is what makes
the resolution auditable rather than merely plausible.


In [ ]:
quality = next(item for item in message_trail if item["type"] == "quality_decision")
review = next(item for item in message_trail if item["type"] == "reviewer_decision")
memories = [entry.content for entry in customer_memory.recall("finance operations lead", limit=20)]

assert quality["status"] == "rejected"
assert set(quality["payload"]["reasons"]) == {"stale knowledge article", "prior customer context omitted"}
assert review["producer"] == "escalation-reviewer"
assert any("finance operations lead" in item.lower() for item in memories)
assert any(event["tool"] == "credit_calculation" for event in tool_events)

show_message_graph(message_trail)
show_spore(next(spore for spore in received_spores if spore.knowledge["type"] == "escalation_request"),
           "Explicit reviewer boundary Spore")
show_json({"states": case_states, "tool_events": tool_events, "durable_memory": memories},
          "Case execution state")
show_timeline()


In [ ]:
show_scorecard([
    ("Routing reached closure", case_states[-1], "pass"),
    ("Stale article rejected", "KB-122", "pass"),
    ("Customer memory used", len(memories), "pass"),
    ("Policy credit", "15% of requested 25%", "pass"),
    ("Reviewer participated", review["payload"]["reviewer"], "pass"),
    ("Revision bounded", 1, "pass"),
])
show_artifact("Auditable closed-case record", case_records[0])


## Failure modes and tradeoffs

The reviewer is deterministic certification input, not a model-generated HITL
interruption. That boundary is named honestly. A production center would also
handle missing service-status data, unavailable entitlement systems, multiple
review rounds, and channel-specific response formatting.


## Extensions

Add a second active incident, a customer without credit entitlement, or an
account preference that conflicts with policy. Keep quality independent from
the resolution coordinator so it can reject a locally convenient answer.

## Cleanup

The in-memory customer store, Agents, Reef workers, and registered tools are
released explicitly.


In [ ]:
customer_memory.memory.clear_agent_memories(customer_memory.name)
customer_memory.close()
for function in agents:
    function._praval_agent.close()
assert reef.shutdown(timeout=10)
reset_tool_registry()
show_callout("Support center closed", "Case resources and workers were released.", role="reef")
